In [122]:
import random
import copy

In [123]:
class Card:
    def __init__(self, suit, value, face_up = False):
        self.suit = suit
        self.value = value
        self.face_up = face_up

    def color(self):
        if self.suit in ["♡", "♢"]:
            return "red"
        else:
            return "black"
        
    def __str__(self):
        if not self.face_up:
            return "▓▓"
        
        value_map = {1: "A", 11: "J", 12: "Q", 13: "K"}

        display_value = value_map.get(self.value, str(self.value));
        return f"{display_value}{self.suit}"

In [124]:
class Deck:
    def __init__(self):
        self.cards = []
        self._build()
        self.shuffle()

    def _build(self):
        suits = ["♡", "♢", "♣", "♠"]
        values = range(1, 14)

        for suit in suits:
            for value in values:
                self.cards.append(Card(suit, value))

    def shuffle(self):
        random.shuffle(self.cards)

    def draw(self):
        if len(self.cards) == 0:
            return None
        return self.cards.pop()

In [125]:
class Pile:
    def __init__(self, pile_type):
        self.cards = []
        self.pile_type = pile_type # tableau, stock, waste, foundation

    def add(self, card):
        self.cards.append(card)

    def remove(self):
        if len(self.cards) == 0: 
            return None
        return self.cards.pop()

    def top(self):
        if len(self.cards) == 0:
            return None
        return self.cards[-1]

In [126]:
class Move:
    def __init__(self, source, dest, count = 1):
        self.source = source
        self.dest = dest
        self.count = count

    def __repr__(self):
        return f"Move({self.count} cards from {self.source} to {self.dest})"

In [127]:
class Game:
    def __init__(self):
        self.history = []
        self.setup()

    # ====== SETUP ======
    def setup(self):
        self.deck = Deck()

        self.tableau = [Pile("tableau") for _ in range(7)]
        self.stock = Pile("stock")
        self.waste = Pile("waste")
        self.foundations = [Pile("foundation") for _ in range(4)]

        for i in range(7):
            for j in range(i+1):
                card = self.deck.draw()
                self.tableau[i].add(card)
            
            top_card = self.tableau[i].top()
            top_card.face_up = True

        while True:
            card = self.deck.draw()
            if card is None:
                break
            self.stock.add(card)
   
    
    # ====== RESTART ======
    def restart(self):
        self.history.clear()
        self.setup()

    def reset_board(self):
        while (self.history):
            self.undo()

    # ====== DEBUGGING HELPERS ======
    def count_all_cards(self):
        total = 0

        for pile in self.tableau:
            total += len(pile.cards)

        total += len(self.stock.cards)
        total += len(self.waste.cards)

        for pile in self.foundations:
            total += len(pile.cards)

        print(f"Total cards: {total}")

    def check_tableau_sizes(self):
        for i, pile in enumerate(self.tableau):
            print(f"Tableau {i}: {len(pile.cards)} cards")

    def check_face_up(self):
        for i, pile in enumerate(self.tableau):
            for j, card in enumerate(pile.cards):
                if j == len(pile.cards) - 1:
                    print(f"Tableau {i} top card face_up =", card.face_up)
                else:
                    print(f"Tableau {i} card {j} face_up =", card.face_up) 

    def check_stock_size(self):
        print(f"Stock size: {len(self.stock.cards)}")      

    def debug_checks(self):
        self.count_all_cards()
        self.check_tableau_sizes()
        self.check_face_up()     
        self.check_stock_size()


    # ====== GAME LOGIC ======
    def draw_from_stock(self):
        # save state before making changes
        # if self.stock.cards or self.waste.cards:
        #     self.save_state()

        # when stock is not empty
        if len(self.stock.cards) > 0: # if self.stock.cards:
            card = self.stock.remove()
            card.face_up = True
            self.waste.add(card)
            return # so that fn returns if this stock is not empty
        
        # when stock is empty
        # as long as waste is not empty you remove from waste face down and then add to stock
        if len(self.waste.cards) > 0: # if self.waste.cards
            while len(self.waste.cards) > 0:
                card = self.waste.remove()
                card.face_up = False
                self.stock.add(card)

    def is_valid_move(self, move):
        # checking if source or dest piles exist
        if move.source is None or move.dest is None:
            return False
        # checking if source is empty
        if len(move.source.cards) < 1: # if not move.source.cards:
            return False
        # checking if either cards to be moved is atleast 1 in number 
        # OR checking if cards to be moved are moved than present in the source pile
        if move.count < 1 or move.count > len(move.source.cards):
            return False
        # checking if all the cards to be moved are face up
        cards_to_move = move.source.cards[-move.count:]

        # for card in cards_to_move:
        #     if not card.face_up:
        #         return False
        # REWRITING THIS USING ALL

        if not all(card.face_up for card in cards_to_move):
            return False
            
        # FIRST RULE YAYYAYAY 
        # okay so first check if its Tableau to Tableau
        # then first rule says compare the bottom of the moving pile and the top of dest pile
        # they must be of oppsoite colors and the top of dest must be exactly one more than the bottom of moving pile
        # so cards_to_move : 0-> bottom and -1-> top ov moving stack
        # TABLEAU TO TABLEAU
        if move.source.pile_type == "tableau" and move.dest.pile_type == "tableau":
            moving_card = cards_to_move[0]
            dest_top = move.dest.top()

            if dest_top is None: # means that dest tableau is empty
                if moving_card.value != 13:
                    return False
                
            else:
                if moving_card.color() == dest_top.color():
                    return False
                if moving_card.value != dest_top.value - 1:
                    return False
                

        # WASTE TO TABLEAU
        if move.source.pile_type == "waste" and move.dest.pile_type == "tableau":
            if move.count != 1:
                return False
            
            waste_card = move.source.top()
            dest_top = move.dest.top()

            if dest_top is None: #means dest tableau is empty
                if waste_card.value != 13:
                    return False
            else:
                if dest_top.color() == waste_card.color():
                    return False
                if waste_card.value != dest_top.value - 1:
                    return False
                
        # FOUNDATION RULES
        # TABLEAU TO FOUNDATION
        if move.dest.pile_type == "foundation" and move.source.pile_type in ("tableau", "waste"):
            if move.count != 1: 
                return False
            dest_top = move.dest.top()
            moving_card = move.source.top()

            if dest_top is None: # meaning dest i.e foundation is empty
                if moving_card.value != 1:
                    return False
            else:
                if moving_card.suit != dest_top.suit:
                    return False
                if moving_card.value != dest_top.value + 1:
                    return False

        return True
    
    def execute_move(self, move):
        if not self.is_valid_move(move):
            return False
        
        cards_to_move = move.source.cards[-move.count:]
        flipped_card = None

        for _ in range(move.count):
            move.source.remove()

        for card in cards_to_move:
            move.dest.add(card)

        if move.source.pile_type == "tableau":
            top_card = move.source.top()
            if top_card is not None and not top_card.face_up:
                top_card.face_up = True
                flipped_card = top_card

        self.history.append({
            "source" : move.source,
            "dest" : move.dest,
            "cards" : cards_to_move,
            "flipped" : flipped_card
        })

        return True
    
    # def is_won(self):
    #     for foundation in self.foundations:
    #         if len(foundation.cards) != 13:
    #             return False
    #     return True

    def is_won(self):
        return all(len(foundation.cards) == 13 for foundation in self.foundations)
    
    # ====== EFFECTIVELY WON =======
    def is_effectively_won(self):
        return all(card.face_up for pile in self.tableau for card in pile.cards)
    #after this check for auto finish condition which is written below under the polishing section

    # ====== POLISHING ======
    # this function is to just check if the auto move to founation is a valid move
    def find_safe_foundation_move(self):
        # tableau to foundation
        # check if tableau top card is Ace or 2 only
        for pile in self.tableau:
            if not pile.cards: # if tableau is empty
                continue

            card = pile.top()
            if card.value not in (1, 2):
                continue
            
            # if 
            for foundation in self.foundations:
                move = Move(pile, foundation, 1) #try moving this tableau card to all foundations and see what works
                if self.is_valid_move(move):
                    return move
            
        # waste to foundation
        if self.waste.cards: # if its not empty 
            for foundation in self.foundations:
                move = Move(self.waste, foundation, 1)
                if self.is_valid_move(move):
                    return move
            
        return None # if its not valid 
    
    # this function is for auto moving to foundation
    def auto_move_to_foundation(self):
        moved_any = False

        while True:
            move = self.find_safe_foundation_move()
            if not move: # meaning there is no valid move
                break

            self.execute_move(move)
            moved_any = True
        
        return moved_any
    

    # ===== UNDO ======
    def undo(self):
        if not self.history: # there is no history
            print("Nothing to undo.")
            return False

        previous = self.history.pop()
        
        source = previous["source"]
        dest = previous["dest"]
        cards = previous["cards"]
        flipped = previous["flipped"]

        for _ in range(len(cards)):
            dest.remove()

        for card in cards:
            source.add(card)

        if flipped:
            flipped.face_up = False

        return True
        

    # ====== AUTO FINISH ======
    def can_auto_finish(self):
        return self.is_effectively_won() and not self.stock.cards and not self.waste.cards
    
    def auto_finish(self):
        while True:
            moved = self.auto_move_to_foundation()
            if not moved:
                break
    


In [128]:
class GamePrinter:
    def __init__(self, game):
        self.game = game
    
    def print_board(self):
        game = self.game

        print("\n-----SOLITAIRE-----\n")

        print("Foundations: ")
        for i, f in enumerate(game.foundations): #go through each foundation pile
            top = f.top()
            if top:
                print(f"  F{i}: {top}")
            else:
                print(f"  F{i}: [empty]")

        print("\nStock / Waste: ")
        print(f"  Stock: {len(game.stock.cards)} cards")
        if game.waste.top(): # i.e waste is not empty
            print(f"  Waste: {game.waste.top()}")
        else:
            print(f"  Waste: [empty]")

        print("\nTableau: ")
        for i, pile in enumerate(game.tableau):
            line = []
            for card in pile.cards:
                line.append(str(card))
            print(f"  T{i}: {' '.join(line)}")

        print()


In [129]:
# class Move:
#     def __init__(self, source, dest, pile):
#         self.source = source
#         self.dest = dest
#         self.pile = pile

#     def valid_move(self):
#         if self.source is None or self.dest is None or self.pile.top() is None:
#             return False
#         if not self.pile.top().face_up:
#             return False
        


#         lorem ipsum

#     def execute_move(self):
#         if self.valid_move():
#             while len(self.pile.cards) > 0:
#                 card = self.pile.remove()
#                 self.dest.add(card)

In [130]:
game = Game()
printer = GamePrinter(game)
printer.print_board()

while True:
    cmd = input("Command (d=draw, m=move, a=auto-move, u=undo, r=restart,n=new-game, q=quit): ").strip().lower()

    if game.can_auto_finish() and not game.is_won():
        print("Auto finish available. Press [f] to finish.")

    if cmd == "q":
        break

    elif cmd == "d":
        game.draw_from_stock()
        printer.print_board()

    elif cmd == "a":
        if game.auto_move_to_foundation():
            print("Auto-moved safe cards to foundation.")
        else:
            print("No safe auto-moves available.")
        printer.print_board()
        
    elif cmd == "u":
        game.undo()
        printer.print_board()

    elif cmd == "r":
        game.reset_board()
        printer.print_board()

    elif cmd == "n":
        game.restart()
        printer.print_board()
    
    elif cmd == "f":
        if game.can_auto_finish():
            game.auto_finish()
            printer.print_board()
            print("\n🎉 YOU WON! 🎉")
            print("[r] Restart   [n] New Game   [q] Quit")
        else:
            print("Auto-finish not available.")
    
    elif cmd.startswith("m"):
        try:
            parts = cmd.split()

            if len(parts) != 4:
                print("Usage: m <source> <dest> <count>")
                continue

            _, src, dst, count_str = parts
            count = int(count_str)
        
            def get_pile(code):
                if code.startswith("t"):
                    return game.tableau[int(code[1])]
                if code.startswith("f"):
                    return game.foundations[int(code[1])]
                if code == "w":
                    return game.waste
                return None

            source = get_pile(src)
            dest = get_pile(dst)

            move = Move(source, dest, count)

            if game.execute_move(move):
                print("Move OK")
            else:
                print("Invalid move")

            printer.print_board()

            if game.is_won():
                print("YOU WON!")
                print("Press [r] to restart or [q] to quit.")
            
        except Exception as e:
            print("Error:", e)



-----SOLITAIRE-----

Foundations: 
  F0: [empty]
  F1: [empty]
  F2: [empty]
  F3: [empty]

Stock / Waste: 
  Stock: 24 cards
  Waste: [empty]

Tableau: 
  T0: Q♠
  T1: ▓▓ 6♢
  T2: ▓▓ ▓▓ 2♡
  T3: ▓▓ ▓▓ ▓▓ 9♠
  T4: ▓▓ ▓▓ ▓▓ ▓▓ A♣
  T5: ▓▓ ▓▓ ▓▓ ▓▓ ▓▓ 9♡
  T6: ▓▓ ▓▓ ▓▓ ▓▓ ▓▓ ▓▓ A♡

